In [ ]:
#Shiven Lahane Notebook for Data UCI Phishing Data Analysis on Phishing Websites
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import sys
import shap

df = pd.read_csv("phishing_websites.csv")
#prints first 5 rows
df.head()

In [ ]:
#Logistic regression model with 15 features vs all features
#Concl
#Split the data → train logistic regression 
# → rank features by coefficient size → keep top 15 → retrain → evaluate.
# 80-20; Using the top 15 features selected by Logistic Regression coefficients, 
#the model achieved 92% accuracy and successfully detected 91% of phishing websites.
# 50-50 no difference -> stable model ig
X = df.drop("result", axis = 1)
y = df["result"].map({-1:0,1:1})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
coeffs = pd.Series(model.coef_[0], index = X_train.columns)

top_15 = coeffs.abs().sort_values(ascending = False).head(15)

top_features = top_15.index

X_train_top = X_train[top_features]
X_test_top = X_test[top_features]

model_top = LogisticRegression(max_iter = 1000)
model_top.fit(X_train_top, y_train)

y_pred = model_top.predict(X_test_top)

print(classification_report(y_test,y_pred))
print(top_15)

In [ ]:
#Using all features, Logistic Regression achieved 92% accuracy,
#catching 90% of phishing sites and correctly identifying 94% of legitimate sites.

#Using all features does NOT improve performance and slightly reduces 
#phishing recall compared to using the top 15 features.
X = df.drop("result", axis = 1)
y = df["result"].map({-1:0,1:1})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
coeffs = pd.Series(model.coef_[0], index = X_train.columns)

y_pred = model.predict(X_test)

print(classification_report(y_test,y_pred))

In [ ]:
#Cross-validation selected a regularized Logistic Regression model (C = 0.1), 
#achieving stable performance with 92% test accuracy and strong phishing detection.
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=2000))
])

param_grid = {"lr__C": [0.01, 0.1, 1, 10, 100]}

grid = GridSearchCV(pipe, param_grid, scoring="accuracy", cv=5)
grid.fit(X_train, y_train)

print("Best C:", grid.best_params_["lr__C"])
print("Best CV accuracy:", grid.best_score_)


best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)


print(classification_report(y_test, y_pred))

In [ ]:
#Using a Random Forest model, websites were classified as phishing or legitimate with 97% accuracy,
# significantly outperforming Logistic Regression; 
#the model shows that phishing websites are primarily characterized by abnormal SSL behavior, 
#suspicious anchor links, low web traffic, excessive subdomains, and unsafe form handling, 
#while legitimate websites consistently exhibit valid SSL certificates, normal linking behavior, 
#higher traffic, and stronger trust signals.
#fit() = learn rules#
#predict() = apply rules
#classification_report() = check how good
#feature_importances_ = which columns mattered most

rf = RandomForestClassifier(n_estimators = 200, random_state =42)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print(classification_report(y_test, y_pred_rf))

rf_importance = pd.Series(rf.feature_importances_, index = X.columns).sort_values(ascending = False)


print(rf_importance.head(15))

In [ ]:
#Random Forest is much safer, cutting missed phishing sites by more than half (94 → 45) 
#and reducing false alarms, making it clearly superior to Logistic Regression for phishing detection.

cm_lr = confusion_matrix(y_test, y_pred)

cm_rf = confusion_matrix(y_test,y_pred_rf)

fig, ax = plt.subplots(1,2, figsize = (10,4))

ConfusionMatrixDisplay(cm_lr, display_labels=["Phishing(0)", "Legit(1)"]).plot(ax=ax[0], values_format="d")
ax[0].set_title("Logistic Regression")

ConfusionMatrixDisplay(cm_rf, display_labels=["Phishing(0)", "Legit(1)"]).plot(ax=ax[1], values_format="d")
ax[1].set_title("Random Forest")

plt.tight_layout()
plt.show()

In [ ]:
#Both models identify SSL validity, suspicious anchor URLs,
# and traffic-based trust signals as the strongest phishing indicators, 
#but Random Forest better captures complex feature interactions, 
#leading to significantly fewer missed phishing websites.

lr_importance = pd.Series(
    abs(model.coef_[0]),
    index=X_train.columns
)

# Random Forest importance (already computed)
rf_importance = rf_importance  # from your previous step

# Combine into one DataFrame (top 15 by RF)
comparison_df = pd.DataFrame({
    "Logistic Regression": lr_importance,
    "Random Forest": rf_importance
}).fillna(0)

comparison_df = comparison_df.sort_values(
    by="Random Forest",
    ascending=False
).head(15)


comparison_df.plot(
    kind="barh",
    figsize=(10, 6)
)

plt.title("Feature Importance Comparison: Logistic Regression vs Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.gca().invert_yaxis()
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
!{sys.executable} -m pip install shap
#The SHAP analysis shows that SSL certificate validity, 
#anchor URL behavior, website traffic, 
#and suspicious domain structure are the strongest indicators of phishing, 
#with invalid SSL, low traffic, and abnormal URLs consistently pushing predictions toward phishing.
explainer = shap.Explainer(rf, X_train)
shap_values = explainer(X_test)

shap.plots.beeswarm(shap_values[:, :, 0]) 

This SHAP beeswarm plot explains how your Random Forest makes phishing decisions by showing, for each feature, how much it pushes a prediction toward phishing (left/negative SHAP values) or toward legitimate (right/positive SHAP values). The features are listed top-to-bottom in order of overall importance, so the top rows influence the model most (like sslfinal_state and url_of_anchor). Each dot represents one website, and its horizontal position shows the strength and direction of that feature’s impact for that specific website. The color shows the feature value (blue = low, red = high), letting you see whether high or low values tend to push the prediction toward phishing or legit. Finally, the “sum of other features” row means the remaining features matter less individually, but together still contribute some influence to the final prediction.